In [2]:
from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=True
)

Mounted at /content/drive


In [3]:
# ==================================================
# 01. CONFIGURACIÓN DEL ENTORNO DE INFERENCIA
# ==================================================

import sys
import torch
import sentencepiece as spm

from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/Proyecto_traductor/traductor_optimizado"
)

MODEL_DIR = BASE_DIR / "modelo"
TOKENIZER_DIR = BASE_DIR / "tokenizador"
CHECKPOINT_DIR = BASE_DIR / "checkpoints"

BEST_CHECKPOINT = CHECKPOINT_DIR / "mejor_modelo.pth"
TOKENIZER_PATH = TOKENIZER_DIR / "tokenizer.model"

# Verificar archivos
archivos = [
    MODEL_DIR / "transformer_model.py",
    TOKENIZER_PATH,
    BEST_CHECKPOINT
]

for archivo in archivos:
    assert archivo.exists(), f"No encontrado: {archivo}"

# Dispositivo
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Cargar checkpoint
checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

MODEL_CONFIG = checkpoint["model_config"]

# Cargar tokenizador
sp = spm.SentencePieceProcessor(
    model_file=str(TOKENIZER_PATH)
)

# Importar arquitectura
if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

from transformer_model import TranslationTransformer

print("=" * 60)
print("ENTORNO DE INFERENCIA")
print("=" * 60)

print("Dispositivo:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print("Mejor modelo - época:", checkpoint["epoch"] + 1)
print("Best Validation Loss:", checkpoint["best_val_loss"])
print("Vocabulario:", sp.vocab_size())
print("Longitud máxima:", MODEL_CONFIG["max_length"])

print("\nEntorno preparado correctamente.")

ENTORNO DE INFERENCIA
Dispositivo: cuda
GPU: Tesla T4
Mejor modelo - época: 15
Best Validation Loss: 2.0290010670697933
Vocabulario: 16000
Longitud máxima: 80

Entorno preparado correctamente.


In [4]:
# ==================================================
# 02. CARGA DEL TRANSFORMER ENTRENADO
# ==================================================

import torch

# Reconstruir arquitectura
model = TranslationTransformer(
    MODEL_CONFIG
).to(DEVICE)

# Cargar pesos del mejor modelo
model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

# Modo inferencia
model.eval()

# Verificar parámetros
total_params = sum(
    p.numel() for p in model.parameters()
)

assert total_params == 19_677_824

print("=" * 60)
print("MODELO CARGADO PARA INFERENCIA")
print("=" * 60)

print("Parámetros:", f"{total_params:,}")
print("Checkpoint:", BEST_CHECKPOINT.name)
print("Época:", checkpoint["epoch"] + 1)
print("Modo evaluación:", not model.training)
print("Dispositivo:", DEVICE)

print("\nModelo listo para traducir.")

MODELO CARGADO PARA INFERENCIA
Parámetros: 19,677,824
Checkpoint: mejor_modelo.pth
Época: 15
Modo evaluación: True
Dispositivo: cuda

Modelo listo para traducir.


In [5]:
# ==================================================
# 03. FUNCIÓN DE TRADUCCIÓN INGLÉS -> ESPAÑOL
# ==================================================

BOS_ID = sp.bos_id()
EOS_ID = sp.eos_id()
PAD_ID = sp.pad_id()
MAX_LENGTH = MODEL_CONFIG["max_length"]


def preparar_entrada(texto):
    tokens = sp.encode(
        texto,
        out_type=int
    )

    tokens = tokens[:MAX_LENGTH - 2]

    tokens = (
        [BOS_ID]
        + tokens
        + [EOS_ID]
    )

    return torch.tensor(
        tokens,
        dtype=torch.long,
        device=DEVICE
    ).unsqueeze(0)


@torch.no_grad()
def traducir(texto, max_length=MAX_LENGTH):
    model.eval()

    src = preparar_entrada(texto)

    # Decoder inicia con BOS
    generated = torch.tensor(
        [[BOS_ID]],
        dtype=torch.long,
        device=DEVICE
    )

    for _ in range(max_length - 1):

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=DEVICE.type == "cuda"
        ):

            logits = model(
                src,
                generated
            )

        next_token = torch.argmax(
            logits[:, -1, :],
            dim=-1
        )

        token_id = next_token.item()

        generated = torch.cat(
            [
                generated,
                next_token.unsqueeze(1)
            ],
            dim=1
        )

        if token_id == EOS_ID:
            break

    tokens_generados = (
        generated.squeeze(0)
        .detach()
        .cpu()
        .tolist()
    )

    tokens_limpios = [
        token
        for token in tokens_generados
        if token not in {BOS_ID, EOS_ID, PAD_ID}
    ]

    traduccion = sp.decode(tokens_limpios)

    return traduccion.strip()


print("=" * 60)
print("FUNCIÓN DE TRADUCCIÓN PREPARADA")
print("=" * 60)
print("Método: Greedy Decoding")
print("BOS:", BOS_ID)
print("EOS:", EOS_ID)
print("PAD:", PAD_ID)
print("Longitud máxima:", MAX_LENGTH)
print("\nFunción traducir() lista.")

FUNCIÓN DE TRADUCCIÓN PREPARADA
Método: Greedy Decoding
BOS: 2
EOS: 3
PAD: 0
Longitud máxima: 80

Función traducir() lista.


In [17]:
# ==================================================
# 04. PRUEBA INTERACTIVA DE TRADUCCIÓN
# ==================================================

texto_ingles = input("Escribe una frase en inglés: ").strip()

if not texto_ingles:
    print("No se ingresó ninguna frase.")
else:
    traduccion = traducir(texto_ingles)

    print("\n" + "=" * 60)
    print("TRADUCCIÓN")
    print("=" * 60)
    print("Inglés :", texto_ingles)
    print("Español:", traduccion)

Escribe una frase en inglés: I went to the park yesterday.

TRADUCCIÓN
Inglés : I went to the park yesterday.
Español: Ayer fui al parque.
